# Evaluation System

This notebook evaluates the VisionGuard AI inspection pipeline.

The goal is to measure:
1. Rule retrieval accuracy
2. Severity classification accuracy
3. Issue detection accuracy
4. Report validity
5. Average latency
6. Human review rate

In [1]:
from pathlib import Path
import sys
import pandas as pd
import json

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from app.backend.services.report_service import InspectionPipelineService

d:\visionguard-ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Evaluation labels

The evaluation labels define expected issue keywords, severity, and expected rule IDs.

In [2]:
EVAL_LABELS_PATH = PROJECT_ROOT / "data" / "eval" / "evaluation_labels.csv"
SAMPLE_IMAGES_DIR = PROJECT_ROOT / "data" / "sample_images"

eval_df = pd.read_csv(EVAL_LABELS_PATH)

eval_df

,image_name,expected_issue_keyword,expected_severity,expected_rule_id
0,ppe_violation_01.jpg,helmet,High,PPE-001
1,helmet_missing_worker.jpg,helmet,High,PPE-001
2,surface_crack_defect.jpg,crack,High,DEF-001
3,metal_crack_component.jpg,crack,High,DEF-001
4,blocked_exit_01.jpg,blocked,High,ESC-003
5,exit_blocked_area.jpg,blocked,High,ESC-003
6,unknown_scene.jpg,unclear,Review Needed,ESC-005
7,blurry_inspection_image.jpg,unclear,Review Needed,ESC-006


## Inspection pipeline

The pipeline runs:

Image → VLM → RAG → LLM Report

In [3]:
pipeline = InspectionPipelineService()

print("Pipeline initialized.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6457.50it/s]


Pipeline initialized.


d:\visionguard-ai\venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


## Evaluation over all test cases

In [4]:
evaluation_results = []

for _, row in eval_df.iterrows():
    image_name = row["image_name"]
    expected_issue_keyword = row["expected_issue_keyword"]
    expected_severity = row["expected_severity"]
    expected_rule_id = row["expected_rule_id"]

    image_path = SAMPLE_IMAGES_DIR / image_name

    if image_path.exists():
        image_bytes = image_path.read_bytes()
    else:
        image_bytes = b"fake-image-bytes"

    result = pipeline.generate_full_inspection_report(
        image_bytes=image_bytes,
        filename=image_name,
        top_k=3,
        enable_tracking=False
    )

    inspection_report = result["inspection_report"]
    retrieved_rules = result["retrieved_rules"]

    retrieved_rule_ids = [
        rule["rule_id"]
        for rule in retrieved_rules
    ]

    issue_type = inspection_report.get("issue_type", "").lower()
    severity = inspection_report.get("severity", "")
    matched_rule_id = inspection_report.get("matched_rule_id", "")
    latency_ms = result.get("latency_ms", None)

    issue_keyword_match = expected_issue_keyword.lower() in issue_type.lower()
    severity_match = severity == expected_severity
    expected_rule_retrieved = expected_rule_id in retrieved_rule_ids
    matched_rule_correct = matched_rule_id == expected_rule_id

    report_valid = all(
        field in inspection_report and inspection_report[field] is not None
        for field in [
            "inspection_id",
            "issue_detected",
            "issue_type",
            "severity",
            "visual_evidence",
            "matched_rule_id",
            "matched_rule_summary",
            "recommended_action",
            "human_review_required",
            "confidence_note",
        ]
    )

    evaluation_results.append({
        "image_name": image_name,
        "expected_issue_keyword": expected_issue_keyword,
        "predicted_issue_type": inspection_report.get("issue_type", ""),
        "issue_keyword_match": issue_keyword_match,
        "expected_severity": expected_severity,
        "predicted_severity": severity,
        "severity_match": severity_match,
        "expected_rule_id": expected_rule_id,
        "retrieved_rule_ids": ", ".join(retrieved_rule_ids),
        "matched_rule_id": matched_rule_id,
        "expected_rule_retrieved": expected_rule_retrieved,
        "matched_rule_correct": matched_rule_correct,
        "human_review_required": inspection_report.get("human_review_required"),
        "latency_ms": latency_ms,
        "report_valid": report_valid,
    })

results_df = pd.DataFrame(evaluation_results)

results_df

,image_name,expected_issue_keyword,predicted_issue_type,issue_keyword_match,expected_severity,predicted_severity,severity_match,expected_rule_id,retrieved_rule_ids,matched_rule_id,expected_rule_retrieved,matched_rule_correct,human_review_required,latency_ms,report_valid
0,ppe_violation_01.jpg,helmet,Missing Helmet,True,High,High,True,PPE-001,"PPE-001, MACH-005, MACH-003",PPE-001,True,True,True,44.51,True
1,helmet_missing_worker.jpg,helmet,Missing Helmet,True,High,High,True,PPE-001,"PPE-001, MACH-005, MACH-003",PPE-001,True,True,True,14.05,True
2,surface_crack_defect.jpg,crack,Surface Crack,True,High,High,True,DEF-001,"DEF-001, DEF-002, DEF-006",DEF-001,True,True,True,50.28,True
3,metal_crack_component.jpg,crack,Surface Crack,True,High,High,True,DEF-001,"DEF-001, DEF-002, DEF-006",DEF-001,True,True,True,21.49,True
4,blocked_exit_01.jpg,blocked,Blocked Emergency Exit,True,High,High,True,ESC-003,"MACH-002, MACH-005, PPE-008",MACH-002,False,False,True,21.80,True
5,exit_blocked_area.jpg,blocked,Blocked Emergency Exit,True,High,High,True,ESC-003,"MACH-002, MACH-005, PPE-008",MACH-002,False,False,True,19.77,True
6,unknown_scene.jpg,unclear,Unclear Visual Evidence,True,Review Needed,High,False,ESC-005,"DEF-001, MACH-008, MACH-006",DEF-001,False,False,True,11.98,True
7,blurry_inspection_image.jpg,unclear,Unclear Visual Evidence,True,Review Needed,High,False,ESC-006,"DEF-001, MACH-008, MACH-006",DEF-001,False,False,True,9.35,True


## Calculate evaluation metrics

In [5]:
metrics = {
    "total_cases": len(results_df),
    "issue_keyword_accuracy": results_df["issue_keyword_match"].mean(),
    "severity_accuracy": results_df["severity_match"].mean(),
    "rule_retrieval_accuracy": results_df["expected_rule_retrieved"].mean(),
    "matched_rule_accuracy": results_df["matched_rule_correct"].mean(),
    "report_validity_rate": results_df["report_valid"].mean(),
    "human_review_rate": results_df["human_review_required"].mean(),
    "average_latency_ms": results_df["latency_ms"].mean(),
}

metrics

{'total_cases': 8,
 'issue_keyword_accuracy': np.float64(1.0),
 'severity_accuracy': np.float64(0.75),
 'rule_retrieval_accuracy': np.float64(0.5),
 'matched_rule_accuracy': np.float64(0.5),
 'report_validity_rate': np.float64(1.0),
 'human_review_rate': np.float64(1.0),
 'average_latency_ms': np.float64(24.15375)}

## Evaluation outputs

In [6]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

EVAL_RESULTS_PATH = REPORTS_DIR / "evaluation_results.csv"
EVAL_SUMMARY_PATH = REPORTS_DIR / "evaluation_summary.md"

results_df.to_csv(EVAL_RESULTS_PATH, index=False)

summary_lines = [
    "# VisionGuard AI Evaluation Summary",
    "",
    "## Metrics",
    "",
]

for key, value in metrics.items():
    if isinstance(value, float):
        summary_lines.append(f"- {key}: {value:.4f}")
    else:
        summary_lines.append(f"- {key}: {value}")

summary_lines.extend([
    "",
    "## Notes",
    "",
    "- Evaluation was performed in mock VLM/LLM mode.",
    "- Mock mode is used to validate the pipeline structure before real API-based VLM evaluation.",
    "- Rule retrieval accuracy may vary because FAISS performs semantic similarity search.",
    "- Human review is intentionally triggered for high-severity and uncertain cases.",
])

EVAL_SUMMARY_PATH.write_text("\n".join(summary_lines), encoding="utf-8")

print("Saved evaluation results to:", EVAL_RESULTS_PATH)
print("Saved evaluation summary to:", EVAL_SUMMARY_PATH)

Saved evaluation results to: D:\visionguard-ai\reports\evaluation_results.csv
Saved evaluation summary to: D:\visionguard-ai\reports\evaluation_summary.md


## Inspect failure cases

This helps us understand where the retrieval or report generation failed.

In [7]:
failures_df = results_df[
    (~results_df["issue_keyword_match"]) |
    (~results_df["severity_match"]) |
    (~results_df["expected_rule_retrieved"]) |
    (~results_df["matched_rule_correct"])
]

failures_df

,image_name,expected_issue_keyword,predicted_issue_type,issue_keyword_match,expected_severity,predicted_severity,severity_match,expected_rule_id,retrieved_rule_ids,matched_rule_id,expected_rule_retrieved,matched_rule_correct,human_review_required,latency_ms,report_valid
4,blocked_exit_01.jpg,blocked,Blocked Emergency Exit,True,High,High,True,ESC-003,"MACH-002, MACH-005, PPE-008",MACH-002,False,False,True,21.80,True
5,exit_blocked_area.jpg,blocked,Blocked Emergency Exit,True,High,High,True,ESC-003,"MACH-002, MACH-005, PPE-008",MACH-002,False,False,True,19.77,True
6,unknown_scene.jpg,unclear,Unclear Visual Evidence,True,Review Needed,High,False,ESC-005,"DEF-001, MACH-008, MACH-006",DEF-001,False,False,True,11.98,True
7,blurry_inspection_image.jpg,unclear,Unclear Visual Evidence,True,Review Needed,High,False,ESC-006,"DEF-001, MACH-008, MACH-006",DEF-001,False,False,True,9.35,True
